In [2]:
import torch
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from peptide_pipeline.generator.cvae_generator import CVAEGenerator


In [3]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

# Import the JSONDataLoader class from the dataloader_json.py in dataloader module
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader

json_loader = JSONDataLoader()
json_loader.load_data("database/training_data.json", columns=["sequence","length","cathionicity","hydrophobicity"])
print(f"Loaded {len(json_loader.data)} peptides from JSON file")
data = json_loader.get_data()

filtered_data = data[(data["length"] >= 5) & (data["length"] <= 14)]
print(f"Filtered peptides (length 5-14): {len(filtered_data)}")

properties = ["length", "cathionicity", "hydrophobicity"]
stats = {}
for prop in properties:
    min_val = float(filtered_data[prop].min())
    max_val = float(filtered_data[prop].max())
    mean_val = float(filtered_data[prop].mean())
    stats[prop] = {"min": min_val, "max": max_val, "mean": mean_val}
    print(f"{prop}: min={min_val:.4f}, max={max_val:.4f}, mean={mean_val:.4f}")

# Scalar conditioning values (baseline-style)
constraints_default = {
    "size": stats["length"]["mean"],
    "molecular_weight": 1500.0,
    "net_charge_pH5_5": 0.0,
    "isoelectric_point": 7.0,
    "hydrophobicity": stats["hydrophobicity"]["mean"],
    "cathionicity": stats["cathionicity"]["mean"],
    "hydrophobic_moment": 0.5,
    "logp": 0.0,
    "boman_index": 0.0,
    "h_bond_donors": 5.0,
    "h_bond_acceptors": 5.0,
    "tpsa": 100.0,
}

#  --- Initialize model ---
cvae = CVAEGenerator(latent_dim=16, hidden_dim=64, condition_dim=32)
print(f"Device:     {cvae.device}")
print(f"Input dim:  {cvae.input_dim}")
print(f"Latent dim: {cvae.latent_dim}")
print(f"Hidden dim: {cvae.hidden_dim}")
print(f"Condition dim: {cvae.condition_dim}")

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=1,
    device=cvae.device,
)
print(f"Condition tensor shape: {conditions.shape}")

MAX_LEN = 14
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
PAD_IDX = 20
VOCAB_SIZE = 21

sequences = [s for s in filtered_data["sequence"].tolist() if 5 <= len(s) <= MAX_LEN]
lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long, device=cvae.device)

def encode_with_pad(seqs, max_len):
    x = torch.zeros(len(seqs), max_len * VOCAB_SIZE)
    for i, seq in enumerate(seqs):
        for j in range(max_len):
            x[i, j * VOCAB_SIZE + PAD_IDX] = 1.0
        for j, aa in enumerate(seq[:max_len]):
            if aa in aa_index:
                x[i, j * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, j * VOCAB_SIZE + aa_index[aa]] = 1.0
    return x

x = encode_with_pad(sequences, MAX_LEN).to(cvae.device)

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=len(sequences),
    device=cvae.device
)

cvae = CVAEGenerator(max_len=MAX_LEN, latent_dim=16, hidden_dim=64, condition_dim=32)
cvae.train_model(data=x, conditions=conditions, lengths=lengths, epochs=100, batch_size=64, lr=1e-3, kl_anneal_epochs=100)

Loaded 4410 peptides from JSON file
Filtered peptides (length 5-14): 3911
length: min=5.0000, max=12.0000, mean=9.7193
cathionicity: min=0.0000, max=12.0000, mean=3.5111
hydrophobicity: min=-3.9400, max=4.5000, mean=0.5059
Device:     cuda
Input dim:  294
Latent dim: 16
Hidden dim: 64
Condition dim: 32
Condition tensor shape: torch.Size([1, 32])
  Epoch 050/100 | Recon: 2.4532 | KL: 0.0000 | KL weight: 0.49
  Epoch 100/100 | Recon: 2.4508 | KL: 0.0000 | KL weight: 0.99


In [4]:
# --- Generate ---
generated = cvae.generate_peptides(count=10, constraints=constraints_default, temperature=0.8)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
expected_len = max(1, min(MAX_LEN, int(round(constraints_default["size"]))))

for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == expected_len
    status = "OK" if valid else "INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")

Peptide 01: GRHRWNRKKV  OK
Peptide 02: WGIVKLRKKL  OK
Peptide 03: RRKRWLRLTL  OK
Peptide 04: QWFRLKDKFI  OK
Peptide 05: RRLRDIKRKF  OK
Peptide 06: LLRWKLKCVI  OK
Peptide 07: FIFRLVPYKA  OK
Peptide 08: VVLSRKARWF  OK
Peptide 09: CLVLLLWLWK  OK
Peptide 10: WVVELLRKRR  OK


In [5]:
# Test with direct scalar constraints and check if generated peptides meet those constraints
constraints = {
    "size": 12,
    "molecular_weight": 1500.0,
    "net_charge_pH5_5": 4.0,
    "isoelectric_point": 10.0,
    "hydrophobicity": 0.10,
    "cathionicity": stats["cathionicity"]["mean"],
    "hydrophobic_moment": 0.5,
    "logp": 0.0,
    "boman_index": 0.0,
    "h_bond_donors": 5.0,
    "h_bond_acceptors": 5.0,
    "tpsa": 100.0,
}

generated = cvae.generate_peptides(
    count=10,
    constraints=constraints,
    temperature=0.8
)

expected_len = int(round(constraints["size"]))
for i, pep in enumerate(generated, 1):
    status = "OK" if len(pep) == expected_len else "INVALID"
    print(f"{i:02d}. {pep}  {status}")

01. VLSLRLQWWLRP  OK
02. SWPTVRKFLVCR  OK
03. KLLAAAARLKKA  OK
04. RKVHWRKRRYRV  OK
05. RWHWLRWTWKRR  OK
06. RFKIKAVWKKKR  OK
07. CLKWRRFRIKRR  OK
08. RKAWLFKWLLFL  OK
09. RLWTRGVRLPGL  OK
10. RRRFKKCLKIKK  OK
